# Inverse medium scattering problem

In the following example we use the iteratively regularized Gauss-Newton solver from `regpy.solvers.nonlinear.irgnm.IrgnmCG` for the example of __medium scattering__.

In medium scattering we try to determine a perturbation $f$ or its corresponding refractive index $1+f$ of a medium from measurements of far ﬁeld patterns $u_{\infty}$ of scattered time-harmonic acoustic waves $u_{sc}:=u - u_{inc}$ in this medium given some incident field $u_{inc}$. The total field $u$ satisfies 

$$
 \Delta u + k^2 (1+f) u = 0 \qquad \text{in } \mathbb{R}^2
$$

The __Iteratively Regularized Gauss-Newton Method (IRGNM)__ minimizes in each iteration 
$$
  f_{n+1}  = f_n + \argmin_{h} \Vert F(f_{n}) + F'[f_n] h - u_{\infty}^{obs}\Vert^{2} + \alpha_{n}  \Vert f_{n} + h - f_0\Vert^{2}
$$
where $F$ is a Fréchet-differentiable operator. The minimum is determined using an implementation of the Tikhonov regularization using a CG method that is implemented as `regpy.solvers.linear.tikhonov.TikhonovCG`.
The regularization parameter $\alpha_n$ in this structure is a decreasing geometric sequence.

## Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

example_dir = "../../../../examples/medium_scattering"
Path(example_dir).resolve()
sys.path.insert(0, str(example_dir))

from regpy import util
from regpy.hilbert import Hm, L2
from regpy.solvers import Setting, Naturals
from mediumscattering import MediumScatteringFixed

## Defining the operator

In [ ]:
# Physical and discretization parameters (unchanged from the original example).
WAVE_NUMBER = 2.0
RADIUS = 1.0
N_INC_WAVES = 16
N_MEAS_DIR = 64
N_GRID = 64

op = MediumScatteringFixed(
    gridshape=(N_GRID, N_GRID),
    radius=RADIUS,
    wave_number=WAVE_NUMBER,
    inc_directions=util.linspace_circle(N_INC_WAVES),
    farfield_directions=util.linspace_circle(N_MEAS_DIR),
)


## Define setting and exact solution

In the regularization we want to have a Sobolev type smoothing penalty. Thus as a penalty we use the $H^2$ space on the domain where we can explicitly pass the support. For the data fidelity we simply use an $L^2$ setting. 

In [ ]:
REL_NOISE_LEVEL = 0.01
NOISE_SEED = 0  # Fix the realization so the example is reproducible.

exact_solution = op.domain.zeros()
r = op.domain.coord_distances()
inside = r < RADIUS
exact_solution[inside] = np.exp(-1.0 / (RADIUS**2 - r[inside]**2))

exact_data = op(exact_solution)

setting = Setting(
    op,
    Hm(mask=op.support, dtype=complex, index=2),
    L2,
    exact_data=exact_data,
    init=op.domain.zeros(),
)
setting.add_Gaussian_noise(
    relative_noise_level=REL_NOISE_LEVEL,
    white_noise=False,
    seed=NOISE_SEED,
)

# The fixed-geometry far field has measurement angles in rows and
# incident directions in columns.
assert exact_data.shape == (N_MEAS_DIR, N_INC_WAVES)


## Plotting data

In [ ]:
def plot_panels(rows, *, row_labels=None, figsize=(10, 5)):
    """Plot rows of (title, 2D array, image extent) with individual colorbars."""
    nrows = len(rows)
    ncols = len(rows[0])
    fig, axes = plt.subplots(
        nrows, ncols, figsize=figsize, constrained_layout=True,
        squeeze=False,
    )
    for i, panels in enumerate(rows):
        if len(panels) != ncols:
            raise ValueError("Each panel row must have the same length.")
        for j, (title, values, bounds) in enumerate(panels):
            ax = axes[i, j]
            image = ax.imshow(values, extent=bounds, origin="lower")
            ax.set_title(title)
            fig.colorbar(image, ax=ax, shrink=0.85)
        if row_labels is not None:
            axes[i, 0].set_ylabel(row_labels[i])
    plt.show()


# imshow uses the second array axis horizontally and the first vertically.
grid = op.domain
extent = (grid.axes[1][0], grid.axes[1][-1],
          grid.axes[0][0], grid.axes[0][-1])
extent_data = (0, 2 * np.pi, 0, 2 * np.pi)

inputs = (
    ("Exact contrast", exact_solution, extent),
    ("Exact far field", exact_data, extent_data),
    ("Noisy far field", setting.data, extent_data),
)
plot_panels(
    [[(title, component(array), bounds) for title, array, bounds in inputs]
     for component in (np.real, np.imag)],
    row_labels=("Real part", "Imaginary part"),
)


## Regularization

Solve the inverse problem with the iteratively regularized Gauss–Newton method.

In [ ]:
MAX_IT = 100
REGPAR_INITIAL = 1e-4
REGPAR_STEP = 2/3  

setting.set_method_params("IRGN_CG", {"regpars": REGPAR_INITIAL * REGPAR_STEP**Naturals()})
reco, reco_data = setting.run("IRGN_CG", max_it=MAX_IT)


## Plotting the results

In [ ]:
# Use the REAL part of the contrast in both panels, as in the
# original notebook (abs(reco) would display negative values differently).
plot_panels([
    [
        ("Exact contrast", exact_solution.real, extent),
        ("Reconstruction", reco.real, extent),
        ("Absolute error", np.abs(reco - exact_solution), extent),
    ],
    [
        ("Exact far field", np.abs(exact_data), extent_data),
        ("Reconstructed far field", np.abs(reco_data), extent_data),
        ("Absolute data error", np.abs(exact_data - reco_data), extent_data),
    ],
])
